# Task 1: Google Play Store Review Scraper & Preprocessing
## 10 Academy Week 2 Challenge: Fintech Customer Experience Analytics

This notebook implements the data engineering pipeline to scrape, clean, and preprocess customer reviews for the three major Ethiopian banking apps:
1. **CBE (Commercial Bank of Ethiopia)** - `prod.cbe.birr`
2. **BOA (Bank of Abyssinia - Apollo)** - `com.boa.apollo`
3. **Dashen Bank (Amole)** - `com.cr2.amolelight`

## Setup & Libraries

In [1]:
import pandas as pd
from google_play_scraper import Sort, reviews
from datetime import datetime
import os
import sys

print("Libraries loaded successfully.")

## Define Ingestion & Scraping Logic
We scrape up to 1500 reviews per bank to ensure a statistically robust baseline, focusing on recent user feedback.

In [2]:
import re
from langdetect import detect

# Common English words/short terms seen in reviews
SHORT_ENGLISH_WORDS = {
    'good', 'excellent', 'nice', 'best', 'perfect', 'bad', 'worst',
    'great', 'love', 'fast', 'slow', 'okay', 'ok', 'super', 'cool',
    'working', 'fine', 'thanks', 'thank', 'you', 'app', 'easy', 'simple',
    'helpful', 'use', 'useful', 'crashes', 'slow', 'lag', 'crashed',
    'error', 'login', 'problem', 'issues', 'issue', 'otp', 'money',
    'transfer', 'trust', 'trustworthy', 'worst', 'poor', 'happy',
    'pleased', 'satisfactory', 'satisfied', 'five', 'star', 'stars',
    'awesome', 'wonderful', 'amazing', 'brilliant', 'fantastic'
}

def is_english(text):
    if not text or not isinstance(text, str):
        return False
    
    text_clean = text.strip()
    if not text_clean:
        return False
        
    # Filter out Ge'ez script (Amharic characters)
    if re.search(r'[\u1200-\u137F\u2D80-\u2DDF\uAB00-\uAB2F]', text_clean):
        return False
        
    # Check if it contains latin characters at all
    if not re.search(r'[a-zA-Z]', text_clean):
        return False
        
    words = [w.lower().strip(".,!?\"'()[]{}*-+=") for w in text_clean.split()]
    words = [w for w in words if w]
    
    if not words:
        return False
        
    if len(words) <= 3:
        if any(w in SHORT_ENGLISH_WORDS for w in words):
            return True
            
    try:
        lang = detect(text_clean)
        if lang == 'en':
            amharic_transliterated_words = {
                'betam', 'arif', 'temetatagn', 'gobez', 'tiru', 'konjo', 'temesgen',
                'ayesram', 'alrisam', 'nw', 'new', 'des', 'yilal', 'yamral', 'yishalal',
                'nechew', 'gar', 'le', 'sew', 'ke', 'na', 'chahn', 'tew'
            }
            amharic_word_count = sum(1 for w in words if w in amharic_transliterated_words)
            if amharic_word_count / len(words) > 0.3:
                return False
            return True
    except Exception:
        pass
        
    english_overlap = sum(1 for w in words if w in SHORT_ENGLISH_WORDS)
    if len(words) > 0 and (english_overlap / len(words)) >= 0.5:
        return True
        
    return False

def scrape_bank_reviews(app_id, bank_name, num_reviews=1500):
    print(f"Scraping reviews for {bank_name} ({app_id})...")
    result = []
    
    try:
        batch, token = reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=num_reviews
        )
        result.extend(batch)
    except Exception as e:
        print(f"Error scraping {bank_name}: {e}")
        return pd.DataFrame()

    # Extract required fields
    data = []
    for r in result:
        row = {
            'id': r.get('reviewId'),
            'review': r.get('content'),
            'rating': r.get('score'),
            'date': r.get('at'),
            'bank': bank_name,
            'source': 'Google Play Store'
        }
        data.append(row)
        
    df = pd.DataFrame(data)
    print(f"Successfully collected {len(df)} raw reviews for {bank_name}.")
    return df

## Execute Data Extraction Pipeline

In [3]:
app_ids = {
    'CBE': 'prod.cbe.birr',
    'BOA': 'com.boa.apollo',
    'Dashen': 'com.cr2.amolelight'
}

dfs = []
for bank, app_id in app_ids.items():
    df_bank = scrape_bank_reviews(app_id, bank, 1500)
    if not df_bank.empty:
        dfs.append(df_bank)

raw_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Total combined raw reviews: {len(raw_df)}")

## Data Cleaning & Preprocessing Steps
1. **De-duplication**: Filter unique reviews using `id`.
2. **Null Filtering**: Drop records with missing text/rating.
3. **Language Filtering**: Keep only English reviews.
4. **Date Standardization**: Normalize dates to `YYYY-MM-DD` ISO format.

In [4]:
def preprocess_data(df):
    if df.empty:
        return df
    
    # 1. Remove duplicates
    initial_count = len(df)
    df = df.drop_duplicates(subset=['id'])
    print(f"Removed {initial_count - len(df)} duplicates.")
    
    # 2. Filter null reviews/ratings
    df = df.dropna(subset=['review', 'rating'])
    
    # 3. Filter English only reviews
    print("Filtering English-only reviews...")
    df = df[df['review'].apply(is_english)]
    
    # 4. Format Date to YYYY-MM-DD
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    
    return df

cleaned_df = preprocess_data(raw_df)
print(f"Total processed and cleaned reviews: {len(cleaned_df)}")

## Save Cleaned Dataset
We serialize the resulting dataset into `data/raw/cleaned_reviews.csv`.

In [5]:
os.makedirs('../data/raw', exist_ok=True)
cleaned_df.to_csv('../data/raw/cleaned_reviews.csv', index=False)
print("Cleaned reviews successfully saved to '../data/raw/cleaned_reviews.csv'.")

## Exploratory Data Summary

In [6]:
print(cleaned_df.groupby('bank')['rating'].describe())
print("\nFirst 5 cleaned records:")
print(cleaned_df.head())